## **TimeGAN**

**TimeGAN** is specifically designed to preserve **temporal dynamics**. It introduces:

### **Static vs. Temporal Features**
- **Static features** (e.g., gender) → do not change over time  
- **Temporal features** (e.g., sensor readings) → vary across time steps

### **Embedding Network**
- An **encoder** and **recovery network** (autoencoder) map data into a latent space  
- Separates **static** and **temporal embeddings**

### **Three Loss Functions**
To enforce realistic temporal transitions:

- **Reconstruction loss (L<sub>R</sub>)** → Ensures faithful reconstruction from the latent space  
- **Unsupervised loss (L<sub>U</sub>)** → Standard GAN adversarial loss  
- **Supervised loss (L<sub>S</sub>)** → Forces the generator to learn step‑wise conditional transitions by forecasting the next embedding

---
 
**TimeGAN** has been tested on datasets with lengths similar to **CMAPSS** (e.g., energy prediction with 24‑hour samples).  

- **Static features** could include the three operating settings:  
  - Altitude  
  - Mach  
  - TRA  

These remain constant across cycles for each engine.


In [1]:
import time_gan_tensorflow.model as model
print(model.__file__)


2026-04-10 11:40:34.054968: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-10 11:40:34.176259: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-10 11:40:34.210442: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-10 11:40:34.896071: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; 

/mnt/c/Users/PC/Downloads/AI/DeepLearning/Data Augmentation/GANs/time_gan_tensorflow/model.py


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time_gan_tensorflow.model import TimeGAN
from time_gan_tensorflow.plots import plot

In [3]:
# import the data
fd001=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/Data PreProcessing/CLEANED_DATA/train/train_FD001_cleaned.csv',sep=',')
fd002=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/Data PreProcessing/CLEANED_DATA/train/train_FD002_cleaned.csv',sep=',')
fd003=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/Data PreProcessing/CLEANED_DATA/train/train_FD003_cleaned.csv',sep=',')
fd004=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/Data PreProcessing/CLEANED_DATA/train/train_FD004_cleaned.csv',sep=',')


In [23]:
fd001.columns

Index(['id', 'cycle', 'RUL', 'setting1', 'setting2', 'setting3', 'sensor_2',
       'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17',
       'sensor_20', 'sensor_21'],
      dtype='object')

In [4]:
test_df=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/FINAL transformer  PROJECT/data cleaning/test_FD001_cleaned.csv',sep=',')
rul_true=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/FINAL transformer  PROJECT/not ready datasets/not ready datasets/RUL_FD001.txt',header=None,names=["RUL"],sep=',')
print(rul_true)
print(fd001['id'])

    RUL
0   112
1    98
2    69
3    82
4    91
..  ...
95  137
96   82
97   59
98  117
99   20

[100 rows x 1 columns]
0          1
1          1
2          1
3          1
4          1
        ... 
20626    100
20627    100
20628    100
20629    100
20630    100
Name: id, Length: 20631, dtype: int64


In [5]:
tets_fd001_cols=[col for col in fd001.columns if not col in ['RUL','setting3']]
tets_df=test_df[tets_fd001_cols]

In [6]:
print(tets_fd001_cols)

['id', 'cycle', 'setting1', 'setting2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


In [7]:
from sklearn.preprocessing import MinMaxScaler
def scaling_process(df):
    y=df['RUL']
    y_clipped = y.clip(upper=125)
    df['RUL']=y_clipped
    feature_cols=[col for col in df.columns if not col in ['id','cycle','setting3']]
    x=df[feature_cols]
    indexes=df[feature_cols]
    x_scaler=MinMaxScaler()
    x_scaled=x_scaler.fit_transform(x)
    scaled_df=pd.DataFrame(
        x_scaled,
        columns=feature_cols,
        index=indexes.index
    )
    scaled_df['id']=df['id']
    return scaled_df,x_scaler

In [8]:
# TimeGAN expects shape (n_samples, sequence_length, n_features)
def windows(df,rul_true=None,seq_len=44):#seq len should be even number
    feature_cols=[col for col in df.columns if not col in ['id','cycle','setting3']]
    X=[]
    y=[]
    
    for engine ,engine_data in df.groupby('id'):
        engine_feats=engine_data[feature_cols].values
        if len(engine_data)>= seq_len:
            if 'RUL' in df.columns:
                engine_rul=engine_data['RUL'].values
            
                for i in range (len(engine_data) - seq_len+1):
                    X.append(engine_feats[i : i + seq_len])
                    
            else:
                
                X.append(engine_feats[- seq_len :])
                y.append(rul_true.loc[engine-1])

        

    return np.array(X),np.array(y)

In [9]:
X1_scaled, x1_scaler = scaling_process(fd001)
X1, y1 = windows(X1_scaled,rul_true ,seq_len=45)
print(X1.shape)  # (num_windows, 45, num_features)
print(y1.shape)  # (num_windows,)


(16231, 45, 17)
(0,)


In [10]:
X1_test,y_test=windows(tets_df,rul_true,seq_len=45)

In [11]:
print (X1_test.shape)

(96, 45, 16)


In [12]:
print(X1_test.shape)

(96, 45, 16)


In [13]:
rul_true=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/FINAL transformer  PROJECT/not ready datasets/not ready datasets/RUL_FD001.txt',header=None,names=["RUL"],sep=',')

y_test

array([[ 98],
       [ 69],
       [ 82],
       [ 91],
       [ 93],
       [ 91],
       [ 95],
       [111],
       [ 96],
       [ 97],
       [124],
       [ 95],
       [107],
       [ 83],
       [ 84],
       [ 50],
       [ 28],
       [ 87],
       [ 16],
       [ 57],
       [113],
       [ 20],
       [145],
       [119],
       [ 66],
       [ 97],
       [ 90],
       [115],
       [  8],
       [ 48],
       [106],
       [  7],
       [ 11],
       [ 19],
       [ 21],
       [ 50],
       [ 28],
       [ 18],
       [ 10],
       [ 59],
       [109],
       [114],
       [ 47],
       [135],
       [ 92],
       [ 21],
       [ 79],
       [114],
       [ 29],
       [ 26],
       [ 97],
       [137],
       [ 15],
       [103],
       [ 37],
       [114],
       [100],
       [ 21],
       [ 54],
       [ 72],
       [ 28],
       [128],
       [ 14],
       [ 77],
       [  8],
       [121],
       [ 94],
       [118],
       [ 50],
       [131],
       [126],
      

In [14]:
X2, y2 = windows(fd002 ,rul_true,seq_len=45)
print(X2.shape)  # (num_windows, 45, num_features)
print(y2.shape)  # (num_windows,)


(42319, 45, 17)
(0,)


In [15]:
X3, y3 = windows(fd003,rul_true ,seq_len=45)
print(X3.shape)  # (num_windows, 45, num_features)
print(y3.shape)  # (num_windows,)


(20320, 45, 17)
(0,)


In [16]:
X4, y4 = windows(fd004, rul_true,seq_len=45)
print(X4.shape)  # (num_windows, 45, num_features)
print(y4.shape)  # (num_windows,)


(50293, 45, 17)
(0,)


In [17]:
# Split the data
x_train=X1
y_train=y1
print(x_train.shape)

# Fit the model to the training data
model = TimeGAN(
    x=x_train,
    timesteps=45,
    hidden_dim=64,
    num_layers=3,
    lambda_param=0.1,
    eta_param=10,
    learning_rate=0.001,
    batch_size=128
)



(16231, 45, 17)


2026-04-10 11:40:40.078039: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:966] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-10 11:40:40.154255: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:966] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-10 11:40:40.154310: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:966] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-10 11:40:40.155432: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow 

In [ ]:
import os
from tensorflow.keras.callbacks import ModelCheckpoint
checkpoint_dir = "./checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, "timegan_epoch_{epoch:02d}.ckpt")

# Train with callback
model.fit(
    epochs=50,
    verbose=True,
   
)

2026-04-10 11:41:14.988455: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] function_optimizer failed: INVALID_ARGUMENT: Input 0 of node zeros_like_38 was passed float from encoder_embedder/gru/PartitionedCall:5 incompatible with expected variant.
2026-04-10 11:41:15.911276: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] function_optimizer failed: INVALID_ARGUMENT: Input 0 of node zeros_like_38 was passed float from encoder_embedder/gru/PartitionedCall:5 incompatible with expected variant.
2026-04-10 11:41:16.416681: W tensorflow/core/common_runtime/process_function_library_runtime.cc:941] Ignoring multi-device function optimization failure: INVALID_ARGUMENT: Input 0 of node zeros_like_38 was passed float from encoder_embedder/gru/PartitionedCall:5 incompatible with expected variant.
2026-04-10 11:41:19.570652: I tensorflow/stream_executor/cuda/cuda_blas.cc:1614] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-04

epoch: 1 autoencoder_loss: 4.040541 generator_loss: 16.758036 discriminator_loss: 1.021460
epoch: 2 autoencoder_loss: 1.302567 generator_loss: 20.122145 discriminator_loss: 1.412730
epoch: 3 autoencoder_loss: 0.566972 generator_loss: 17.263027 discriminator_loss: 1.096377
epoch: 4 autoencoder_loss: 0.229093 generator_loss: 18.191330 discriminator_loss: 2.300734
epoch: 5 autoencoder_loss: 0.170588 generator_loss: 13.875029 discriminator_loss: 1.300384
epoch: 6 autoencoder_loss: 0.150898 generator_loss: 11.415070 discriminator_loss: 1.360542
epoch: 7 autoencoder_loss: 0.119913 generator_loss: 10.662399 discriminator_loss: 1.964491
epoch: 8 autoencoder_loss: 0.097959 generator_loss: 12.016794 discriminator_loss: 1.683392
epoch: 9 autoencoder_loss: 0.091553 generator_loss: 8.690796 discriminator_loss: 1.213171
epoch: 10 autoencoder_loss: 0.065237 generator_loss: 9.365296 discriminator_loss: 1.226615
epoch: 11 autoencoder_loss: 0.056144 generator_loss: 7.699661 discriminator_loss: 1.666741


KeyboardInterrupt: 

In [20]:
# Save all three models' weights
save_path = "./timegan_checkpoint"
model.autoencoder_model.save_weights(f"{save_path}_autoencoder.h5")
model.generator_model.save_weights(f"{save_path}_generator.h5")
model.discriminator_model.save_weights(f"{save_path}_discriminator.h5")
print("Weights saved .")

Weights saved .


In [21]:
# Recreate the exact same architecture
model = TimeGAN(
    x=x_train,
    timesteps=45,
    hidden_dim=64,
    num_layers=3,
    lambda_param=0.1,
    eta_param=10,
    learning_rate=0.001,
    batch_size=128)

# Load the weights
model.autoencoder_model.load_weights("./timegan_checkpoint_autoencoder.h5")
model.generator_model.load_weights("./timegan_checkpoint_generator.h5")
model.discriminator_model.load_weights("./timegan_checkpoint_discriminator.h5")

In [ ]:
import numpy as np

extra_feature = np.zeros((96, 45, 1))
extra_feature[:, -1, 0] = y_test[:, 0]
x_test = np.concatenate([X1_test, extra_feature], axis=2)

print(x_test.shape)


(96, 45, 17)


In [ ]:
x_hat = model.reconstruct(x=x_test)


2026-04-10 19:35:57.713129: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8101


In [ ]:
# Generate the synthetic data
x_sim = model.simulate(num_sequences=len(x_test))

In [ ]:
# Plot the actual, reconstructed and synthetic data
import kaleido
kaleido.get_chrome_sync()

fig = plot(actual=X1_test, reconstructed=x_hat, synthetic=x_sim)
fig.write_image("results.png", scale=4, height=900, width=700)
fig.show()

In [32]:
print(f"x_hat shape: {x_hat.shape}")
print(f"x_sim shape: {x_sim.shape}")

x_hat shape: (4320, 17)
x_sim shape: (96, 45, 17)


In [ ]:
import numpy as np

np.save('reconstructed_samples.npy', x_hat)
np.save('synthetic_samples.npy', x_sim)
print("Saved to reconstructed_samples.npy and synthetic_samples.npy")

Saved to reconstructed_samples.npy and synthetic_samples.npy


In [ ]:
# Reshape 2D to 3D (assuming the length is a multiple of timesteps)
total_length = x_sim.shape[0]
timesteps = 45
samples = total_length // timesteps
features = x_sim.shape[1]

x_sim_3d = x_sim.reshape(samples, timesteps, features)
print(f"Reshaped to 3D: {x_sim_3d.shape}")

# Save the 3D version
np.save('synthetic_windowed.npy', x_sim_3d)

Reshaped to 3D: (2, 45, 16)


In [ ]:
import pandas as pd
feature_cols=[col for col in fd001.columns if not col in ['id','cycle','RUL','setting3']]
print(len(feature_cols))
# Create a DataFrame (each column is a feature)
df_sim = pd.DataFrame(x_sim,columns=feature_cols)
df_sim.to_csv('synthetic_timeseries.csv', index=False)
print("Saved as CSV.")

16
Saved as CSV.
